# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LiquidMercury-tech/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline is a simple opportunity score for pages that already have meaningful visibility but under-capture clicks or engagement. I score pages higher when they have high impressions, visible ranking positions, low CTR relative to their position tier, and weak engagement. The reason codes are `high_impression_visible_page`, `low_ctr_visible_page`, `weak_engagement_page`, `stale_visible_page`, and `monitor_only` for low-volume edge cases.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_data_path():
    starts = [Path.cwd()]
    for _ in range(8):
        starts.append(starts[-1].parent)
    for root in starts:
        candidate = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(find_data_path())
df = df.copy()
df['visible'] = (df['avg_position'] > 0) & (df['avg_position'] <= 20)
df['high_volume'] = df['impressions_90d'] >= 500
df['low_ctr'] = df['visible'] & (df['ctr'] < 0.5)
df['weak_engagement'] = (df['sessions_90d'] >= 30) & ((df['engagement_rate'] < 30) | (df['scroll_rate'] < 30))
df['stale_visible'] = (df['content_age_days'] >= 180) & df['high_volume'] & df['visible']
visibility_score = np.clip(df['impressions_90d'] / df['impressions_90d'].quantile(0.95), 0, 1)
ctr_gap = np.clip((0.5 - df['ctr']) / 0.5, 0, 1)
engagement_gap = np.clip((30 - df[['engagement_rate', 'scroll_rate']].min(axis=1)) / 30, 0, 1)
df['baseline_score'] = 100 * (0.45 * visibility_score + 0.35 * np.where(df['visible'], ctr_gap, 0) + 0.20 * np.where(df['weak_engagement'], engagement_gap, 0))
df['baseline_score'] = df['baseline_score'].clip(0, 100)
df['reason_code'] = np.select([df['stale_visible'] & df['high_volume'], df['low_ctr'], df['weak_engagement'], df['high_volume'] & df['visible'], (df['impressions_90d'] < 500) | (~df['visible'])], ['stale_visible_page', 'low_ctr_visible_page', 'weak_engagement_page', 'high_impression_visible_page', 'monitor_only'], default='monitor_only')
queue = df[['content_id', 'client_id', 'baseline_score', 'reason_code', 'impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'content_age_days']].copy()
queue = queue.sort_values('baseline_score', ascending=False).reset_index(drop=True)
queue['rank'] = np.arange(1, len(queue) + 1)
queue['action'] = np.select([queue['reason_code'].eq('low_ctr_visible_page'), queue['reason_code'].eq('weak_engagement_page'), queue['reason_code'].eq('stale_visible_page'), queue['reason_code'].eq('high_impression_visible_page')], ['rewrite title/meta to improve click-through', 'improve on-page engagement and scannability', 'refresh stale content and update metadata', 'monitor and test a CTR or metadata refresh'], default='monitor only')
out_path = Path('work/outputs/baseline_action_score.csv')
out_path.parent.mkdir(exist_ok=True, parents=True)
queue.to_csv(out_path, index=False)
print(f'Wrote {len(queue)} rows to {out_path}')
print(queue.head(5).to_string(index=False))


Wrote 30000 rows to work\outputs\baseline_action_score.csv
          content_id         client_id  baseline_score          reason_code  impressions_90d  ctr  avg_position  engagement_rate  scroll_rate  content_age_days  rank                                      action
content_dd635253d90e client_6208ef0f77       98.600000 low_ctr_visible_page            33286 0.02           4.6             0.00        10.09               139     1 rewrite title/meta to improve click-through
content_8330b826f55b client_3fdba35f04       98.600000 low_ctr_visible_page            24808 0.02          16.5             0.00        29.38               165     2 rewrite title/meta to improve click-through
content_f203a581de3d client_f74efabef1       97.953333 low_ctr_visible_page            24242 0.02          10.0             0.97         1.94               125     3 rewrite title/meta to improve click-through
content_42634cb0c5a3 client_6208ef0f77       97.913333   stale_visible_page            43175 0.02    

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The code above writes a ranked candidate queue and makes the score transparent. The benefit of this baseline is that it is easy to explain, and any learned model should beat it on the same review queue. We measure the baseline with the same top-K precision test used for the model later.

In [2]:
queue = pd.read_csv('work/outputs/baseline_action_score.csv')
top20 = queue.head(20).copy()
top20['confidence_note'] = np.where(top20['reason_code'].isin(['low_ctr_visible_page', 'weak_engagement_page']), 'medium confidence: observed gap plus enough demand to matter', 'lower confidence: watch for volume/noise issues')
print('Top 20 queue preview:')
print(top20[['rank', 'content_id', 'baseline_score', 'reason_code', 'action']].head(10).to_string(index=False))


Top 20 queue preview:
 rank           content_id  baseline_score          reason_code                                      action
    1 content_dd635253d90e       98.600000 low_ctr_visible_page rewrite title/meta to improve click-through
    2 content_8330b826f55b       98.600000 low_ctr_visible_page rewrite title/meta to improve click-through
    3 content_f203a581de3d       97.953333 low_ctr_visible_page rewrite title/meta to improve click-through
    4 content_42634cb0c5a3       97.913333   stale_visible_page   refresh stale content and update metadata
    5 content_033d8356c2d9       97.900000 low_ctr_visible_page rewrite title/meta to improve click-through
    6 content_9c8299b55f3c       97.900000   stale_visible_page   refresh stale content and update metadata
    7 content_4a6607efcb46       97.766667 low_ctr_visible_page rewrite title/meta to improve click-through
    8 content_50a9f9f861c6       97.200000   stale_visible_page   refresh stale content and update metadata
    9 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The strongest picks sit at the intersection of real demand, visible rankings, and a measured underperformance. A page at page 1-3 with strong impressions and poor CTR is an obvious candidate for metadata or title work. A page with high sessions but weak engagement is a candidate for depth, UX, or content structure improvements. The pages I would be most skeptical about are the ones with high score because of a single strong signal but thin evidence, or very low volume that makes the result noisy.

In [3]:
queue = pd.read_csv('work/outputs/baseline_action_score.csv')
top20 = queue.head(20).copy()
top20['why_it_might_be_wrong'] = np.where(top20['reason_code'].eq('low_ctr_visible_page'), 'CTR may be acceptable for this position tier or the page may be a seasonal topic', 'The issue could be low volume, a query mix change, or page intent mismatch')
print(top20[['rank', 'content_id', 'reason_code', 'action', 'why_it_might_be_wrong']].to_string(index=False))


 rank           content_id          reason_code                                      action                                                            why_it_might_be_wrong
    1 content_dd635253d90e low_ctr_visible_page rewrite title/meta to improve click-through CTR may be acceptable for this position tier or the page may be a seasonal topic
    2 content_8330b826f55b low_ctr_visible_page rewrite title/meta to improve click-through CTR may be acceptable for this position tier or the page may be a seasonal topic
    3 content_f203a581de3d low_ctr_visible_page rewrite title/meta to improve click-through CTR may be acceptable for this position tier or the page may be a seasonal topic
    4 content_42634cb0c5a3   stale_visible_page   refresh stale content and update metadata       The issue could be low volume, a query mix change, or page intent mismatch
    5 content_033d8356c2d9 low_ctr_visible_page rewrite title/meta to improve click-through CTR may be acceptable for this position tie

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The main weak picks are pages with high score but low volume or with otherwise healthy engagement. I also explicitly exclude product-generated flags or future-window target information. The dataset does not ship product scores, and the score uses only observed 90-day signals that were available before the review decision. That keeps the baseline decision-support, not a hidden replay of FlyRank's own rules.

In [4]:
import pandas as pd
df = pd.read_csv(find_data_path())
queue = pd.read_csv('work/outputs/baseline_action_score.csv')
weak = queue[(queue['baseline_score'] > 60) & ((queue['impressions_90d'] < 500) | (queue['avg_position'].isna()))].head(10)
print('Weak/edge candidates likely to be noisy:')
print(weak[['content_id', 'baseline_score', 'reason_code', 'impressions_90d', 'avg_position']].to_string(index=False))
print('Leakage check: no product flags present in starter data; feature window is 90-day observed search and engagement data only.')
print(f'Rows with avg_position=0: {int((df['avg_position'] == 0).sum())}; these are excluded from rank-based review scoring.')


Weak/edge candidates likely to be noisy:
Empty DataFrame
Columns: [content_id, baseline_score, reason_code, impressions_90d, avg_position]
Index: []
Leakage check: no product flags present in starter data; feature window is 90-day observed search and engagement data only.
Rows with avg_position=0: 1205; these are excluded from rank-based review scoring.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.